## Modéliser et prevoir le risque assurantiel à partir de données historiques de sinistres

## ⚙️ Modèle Linéaire Généralisé (GLM): Poisson (fréquence), Gamma (sévérité), Tweedie (prime pure)

### ➡️ Chargement des librairies 

In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import PoissonRegressor
from sklearn.linear_model import GammaRegressor
from sklearn.linear_model import TweedieRegressor

### ➡️ Dataset

In [7]:
data = pd.read_csv("Data/freMTPL2freq_S15.csv")

data.head()

,IDpol,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region
0,1.0,1,0.10,D,5,0,55,50,B12,Regular,1217,R82
1,3.0,1,0.77,D,5,0,55,50,B12,Regular,1217,R82
2,5.0,1,0.75,B,6,2,52,50,B12,Diesel,54,R22
3,10.0,1,0.09,B,7,0,46,50,B12,Diesel,76,R72
4,11.0,1,0.84,B,7,0,46,50,B12,Diesel,76,R72


### ➡️ Préparation des données

In [12]:
# Variable cible : ClaimNb (fréquence)
y = data["ClaimNb"]

# Variables explicatives
X = data.drop(columns=["ClaimNb"])

# Colonnes catégorielles
cat_cols = ["Area","VehBrand","VehGas","Region"]

# Colonnes numériques
num_cols = ["Exposure","VehPower","VehAge","DrivAge",
            "BonusMalus","Density"]

# Préprocesseur
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols)
    ]
)

### ➡️ Modèle GLM Poisson

In [8]:
model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("glm", PoissonRegressor(alpha=1e-4))
])

# Entraînement
model.fit(X, y)

# Prédiction
pred = model.predict(X)
print("Fréquence prédite :", pred[0])

Fréquence prédite : 0.053246766654916636


### ➡️ Modèle Gamma - Sévérité des sinistres

In [13]:
np.random.seed(0)  
data["ClaimCost"] = np.where(
    data["ClaimNb"] > 0,
    np.random.gamma(shape=2, scale=1000, size=len(data)),  # coûts simulés
    0
)    

data_sev = data[data["ClaimNb"] > 0].copy()
data_sev["Severity"] = data_sev["ClaimCost"] / data_sev["ClaimNb"]

X_sev = data_sev.drop(columns=["Severity", "ClaimCost"])
y_sev = data_sev["Severity"]

In [14]:
sev_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("glm_gamma", GammaRegressor(alpha=1e-4))
])

sev_model.fit(X_sev, y_sev)
sev_pred = sev_model.predict(X_sev)
print("Sévérité prédite :", sev_pred[0])

Sévérité prédite : 1946.104474128466


### ➡️ Prime Pure (Modèle Tweedie)

In [16]:
# Prime pure = ClaimCost / Exposure
data["PurePremium"] = data["ClaimCost"] / data["Exposure"]

X_pp = data.drop(columns=["PurePremium", "ClaimCost"])
y_pp = data["PurePremium"]

In [17]:
pp_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("glm_tweedie", TweedieRegressor(power=1.5, alpha=1e-4))
])

pp_model.fit(X_pp, y_pp)
pp_pred = pp_model.predict(X_pp)
print("Prime pure prédite :", pp_pred[0])

c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\core\fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


Prime pure prédite : 496.7347907287371


### ➡️ Résumé

Cette analyse menée à partir des modèles GLM permet d’obtenir une vision structurée et cohérente du risque assurantiel.
Le modèle Poisson fournit une fréquence prédite de 0.053, indiquant une probabilité relativement faible de survenance d’un sinistre pour le profil étudié.
En supposant ou en simulant un coût moyen de sinistre, le modèle Gamma permet d’estimer une sévérité moyenne de 1 946 €, représentant le coût attendu d’un sinistre lorsqu’il survient.
La combinaison de ces deux composantes via un modèle Tweedie conduit à une prime pure prédite de 496.73 €, correspondant au coût théorique attendu par unité d’exposition.

Ces résultats illustrent la complémentarité des modèles GLM dans la décomposition du risque  :

Poisson pour la probabilité d’occurrence,

Gamma pour le coût moyen et

Tweedie pour la prime pure.

Ensemble, ils offrent une approche robuste et interprétable pour la modélisation du risque assurantiel et constituent une base solide pour la tarification, la segmentation et l’analyse du portefeuille.